# RGCN Standalone Test Notebook

**Purpose:** Test RGCN training in isolation without running all cells from the main notebook.

Once this works, copy the thread safety fix to the main notebook.

In [1]:
# CELL 1: Thread Safety (MUST run first!)
# This prevents macOS segfaults and hangs
import os
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'

import sys
sys.path.insert(0, '..')

import torch
torch.set_num_threads(1)

import numpy as np
import time

print(f'PyTorch: {torch.__version__}')
print(f'Threads: {torch.get_num_threads()}')
print('✓ Thread safety configured')

PyTorch: 2.10.0
Threads: 1
✓ Thread safety configured


In [2]:
# CELL 2: Import SIMPLE RGCN (avoids PyG crashes)
# Using custom implementation that doesn't use RGCNConv
from src.models.rgcn_simple import train_simple_rgcn, SimpleRGCN
print('✓ Simple RGCN module imported (no PyG RGCNConv)')

✓ RGCN module imported


In [ ]:
# CELL 2b: Test PyTorch operations BEFORE loading data
# This catches crashes early before wasting time on data loading

print("Testing PyTorch operations...")

try:
    # Test 1: Basic tensor creation
    print("  [1/5] Tensor creation...", end=" ")
    x = torch.randn(100, 12)
    print("✓")
    
    # Test 2: Linear layer
    print("  [2/5] Linear layer...", end=" ")
    layer = torch.nn.Linear(12, 64)
    out = layer(x)
    print("✓")
    
    # Test 3: index_add_ (used in message passing)
    print("  [3/5] index_add_ (scatter)...", end=" ")
    src = torch.randn(50, 64)
    idx = torch.randint(0, 100, (50,))
    dst = torch.zeros(100, 64)
    dst.index_add_(0, idx, src)
    print("✓")
    
    # Test 4: Backward pass
    print("  [4/5] Backward pass...", end=" ")
    x = torch.randn(100, 12, requires_grad=True)
    out = torch.nn.Linear(12, 1)(x).sum()
    out.backward()
    print("✓")
    
    # Test 5: Mini forward pass with SimpleRGCN
    print("  [5/5] SimpleRGCN forward...", end=" ")
    from src.models.rgcn_simple import SimpleRGCN
    model = SimpleRGCN(num_features=12, hidden_channels=32, num_layers=2)
    x = torch.randn(100, 12)
    edge_index = torch.randint(0, 100, (2, 200))
    edge_type = torch.randint(0, 2, (200,))
    with torch.no_grad():
        out = model(x, edge_index, edge_type)
    print("✓")
    
    print("\n✅ All PyTorch operations working! Safe to proceed.")
    PYTORCH_SAFE = True
    
except Exception as e:
    print(f"\n❌ FAILED: {e}")
    print("PyTorch operations crashed - do not proceed with training.")
    PYTORCH_SAFE = False
    import traceback
    traceback.print_exc()

In [ ]:
# CELL 3: Load REAL data from database
import pandas as pd
from pathlib import Path
from src.core.cve_database import CVEDatabase
from src.features.engineering import create_all_features

# Connect to database
db_path = Path('../data/cve_database.db')
db = CVEDatabase(str(db_path))

# Load CVE data (same query as main notebook)
query = """
SELECT 
    c.cve_id, c.published, c.cvss, c.cwe,
    e.epss_score, e.epss_percentile, e.kev_flag, e.is_healthcare,
    e.attack_flag, e.attack_technique_count, e.chpl_flag, e.label
FROM cves c
LEFT JOIN enrichments e ON c.cve_id = e.cve_id
WHERE c.published >= '2024-01-01'
    AND c.cvss IS NOT NULL
    AND e.label IS NOT NULL
ORDER BY c.published DESC
"""

df = pd.read_sql(query, db.conn)
df['published'] = pd.to_datetime(df['published'])
df['has_cwe'] = df['cwe'].notna() & (df['cwe'] != '')
df['cwe_primary'] = df['cwe'].str.extract(r'(CWE-\d+)')[0]

# Temporal split
split_date = pd.Timestamp('2024-11-01')
train_df = df[df['published'] < split_date].copy().reset_index(drop=True)

# Feature columns (same as main notebook)
feature_cols = [
    'cvss_norm', 'epss_score', 'epss_percentile', 'kev_flag',
    'days_since_published', 'recency_score', 'attack_technique_count', 'has_attack',
    'chpl_flag', 'is_healthcare', 'cvss_epss_product', 'kev_healthcare_interaction'
]

# Create features
train_features_df = create_all_features(train_df, feature_cols)
features = train_features_df[feature_cols].values.astype(np.float32)
labels = train_df['label'].values.astype(np.float32)

# Build CVE-CWE mapping (same logic as main notebook)
MAX_NEIGHBORS = 5
cve_to_cwe = {}
for idx, row in train_df.iterrows():
    if pd.notna(row.get('cwe_primary')):
        same_cwe_cves = train_df[train_df['cwe_primary'] == row['cwe_primary']].index.tolist()
        cve_to_cwe[idx] = [i for i in same_cwe_cves if i != idx][:MAX_NEIGHBORS]
    else:
        cve_to_cwe[idx] = []

# Train/val split
N = len(train_df)
train_size = int(0.8 * N)
train_idx = np.arange(train_size)
val_idx = np.arange(train_size, N)

print(f'✓ REAL data loaded from database:')
print(f'  Total CVEs: {len(df):,}')
print(f'  Training CVEs: {N:,}')
print(f'  Features shape: {features.shape}')
print(f'  Train: {len(train_idx):,}, Val: {len(val_idx):,}')
print(f'  CVEs with CWE connections: {sum(1 for v in cve_to_cwe.values() if len(v) > 0):,}')
print(f'  Total edges: {sum(len(v) for v in cve_to_cwe.values()) * 2:,}')

2026-01-27 23:30:35 - src.core.cve_database - INFO - Connected to database
2026-01-27 23:30:35 - src.core.cve_database - INFO - Database schema created/verified
Feature engineering complete: 32,286 rows, 12 features

Feature statistics:
                                mean      std       min       max
cvss_norm                     0.6683   0.1685    0.0000    1.0000
epss_score                    0.0232   0.1100    0.0000    0.9458
epss_percentile               0.4173   0.2700    0.0001    1.0000
kev_flag                      0.0039   0.0621    0.0000    1.0000
days_since_published        605.6331  84.3441  453.0000  757.0000
recency_score                 0.2000   0.1114    0.0000    0.4016
attack_technique_count        0.6155   0.8755    0.0000    8.0000
has_attack                    0.4210   0.4937    0.0000    1.0000
chpl_flag                     0.0000   0.0000    0.0000    0.0000
is_healthcare                 0.6331   0.4820    0.0000    1.0000
cvss_epss_product             0.0192 

: 

In [ ]:
# CELL 4: Train SIMPLE RGCN with error handling
import time
import traceback

if not PYTORCH_SAFE:
    print("❌ Skipping - PyTorch safety check failed in Cell 2b")
else:
    print('='*60)
    print('TRAINING SIMPLE RGCN (with error handling)')
    print('='*60)
    print(f'Samples: {len(features):,}')
    print(f'Expected: ~10-20 seconds\n')

    t0 = time.time()
    
    try:
        model, trainer, history = train_simple_rgcn(
            cve_features=features,
            cve_to_cwe=cve_to_cwe,
            cve_labels=labels,
            train_idx=train_idx,
            val_idx=val_idx,
            hidden_channels=64,
            num_layers=2,
            dropout=0.2,
            learning_rate=0.01,
            epochs=100,
            early_stopping_patience=10,
            verbose=True
        )

        elapsed = time.time() - t0
        print(f'\n{"="*60}')
        print(f'✓ TRAINING COMPLETE!')
        print(f'  Time: {elapsed:.1f}s')
        print(f'  Epochs: {len(history["train_loss"])}')
        print(f'  Final train loss: {history["train_loss"][-1]:.4f}')
        print(f'  Final val loss: {history["val_loss"][-1]:.4f}')
        print(f'{"="*60}')

        if elapsed < 60:
            print('\n🎉 SUCCESS! No crashes. Ready to apply to main notebook.')
        else:
            print('\n⚠ Completed but took longer than expected.')
            
    except RuntimeError as e:
        print(f'\n❌ RuntimeError during training: {e}')
        traceback.print_exc()
        print('\nThis is likely a PyTorch/memory issue.')
        
    except Exception as e:
        print(f'\n❌ Error during training: {e}')
        traceback.print_exc()

TRAINING RGCN (32K samples, 100 epochs)
Expected: ~10 seconds with early stopping


RGCN TRAINING PIPELINE

[STEP 1/4] Preparing data...
Preparing RGCN data (32,286 nodes)...
  [1/4] Normalizing features... ✓
  [2/4] Building edges (32,286 mappings)... ✓ (286,812 edges)
  [3/4] Creating tensors... ✓
  [4/4] Creating masks... ✓
  → Nodes: 32,286, Edges: 286,812, Train/Val: 25828/6458

[STEP 2/4] Setting up device...
  → Using CPU (faster than MPS for sparse ops)
  → Mini-batch enabled (32,286 nodes > 5K)
  → Batch size: 1024

[STEP 3/4] Creating model...
  → Model: 12 → 64 → 1
2026-01-27 23:31:01 - src.models.rgcn - INFO - RGCN Trainer initialized on device: cpu
2026-01-27 23:31:01 - src.models.rgcn - INFO - Mini-batch training: True, batch_size: 1024

[STEP 4/4] Training...
⚠ NeighborLoader not available (ImportError), using full-batch
Training: [

## ✅ If Cell 4 completed successfully

The fix works! To apply it to your main notebook:

1. In **CVE_Prioritization_Advanced.ipynb**, find cell 29 (RGCN imports)
2. Add these lines at the **TOP** of that cell:

```python
import os
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'

import torch
torch.set_num_threads(1)
```

3. Restart kernel and run all cells - RGCN training should complete in ~10s